# 3st week's homework - by [Aleksei Novikov](https://www.linkedin.com/in/devnovikov/)

## Preparation

In [229]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

import seaborn as sns

%matplotlib inline

## Data preparation


* Check if the missing values are presented in the features.
* If there are missing values:
    * For caterogiral features, replace them with 'NA'
    * For numerical features, replace with with 0.0

In [230]:
df = pd.read_csv("./course_lead_scoring.csv")

In [231]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)
numeric_columns = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

for col in categorical_columns:
    df[col] = df[col].str.lower().str.replace(' ', '_')

In [232]:
df.columns

Index(['lead_source', 'industry', 'number_of_courses_viewed', 'annual_income',
       'employment_status', 'location', 'interaction_count', 'lead_score',
       'converted'],
      dtype='object')

In [233]:
df.isnull().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [234]:
numeric_columns

['number_of_courses_viewed',
 'annual_income',
 'interaction_count',
 'lead_score',
 'converted']

In [235]:
df[categorical_columns] = df[categorical_columns].fillna("NA")
df[numeric_columns] = df[numeric_columns].fillna(0.0)

df.isnull().sum()

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

### Question 1

What is the most frequent observation (mode) for the column `industry`?

- `retail`

In [236]:
mode_value = df["industry"].mode()[0]
mode_value

'retail'

### Question 2

Create the [correlation matrix](https://www.google.com/search?q=correlation+matrix) for the numerical features of your dataset.
In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- `annual_income` and `interaction_count`

Only consider the pairs above when answering this question.

### Split the data

* Split your data in train/val/test sets with 60%/20%/20% distribution.
* Use Scikit-Learn for that (the `train_test_split` function) and set the seed to `42`.
* Make sure that the target value `y` is not in your dataframe.


In [237]:
correlation_matrix = df[numeric_columns].corr()

pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

correlations = {}
for feat1, feat2 in pairs:
    corr_value = correlation_matrix.loc[feat1, feat2]
    correlations[(feat1, feat2)] = corr_value
    print(f"{feat1} and {feat2}: {corr_value:.4f}")


max_pair = max(correlations, key=correlations.get)
print(f"Highest correlation: {max_pair[0]} and {max_pair[1]} ({correlations[max_pair]:.4f})")

interaction_count and lead_score: 0.0099
number_of_courses_viewed and lead_score: -0.0049
number_of_courses_viewed and interaction_count: -0.0236
annual_income and interaction_count: 0.0270
Highest correlation: annual_income and interaction_count (0.0270)


In [238]:
df_train_full, df_test = train_test_split(df, test_size=0.2,  random_state=42, shuffle=True)
df_train, df_val = train_test_split(df_train_full, test_size=0.25,  random_state=42, shuffle=True)
len(df_train), len(df_val), len(df_test)

df_train_full = df_train_full.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print(f"Train set size: {len(df_train)} ({len(df_train)/len(df)*100:.1f}%)")
print(f"Validation set size: {len(df_val)} ({len(df_val)/len(df)*100:.1f}%)")
print(f"Test set size: {len(df_test)} ({len(df_test)/len(df)*100:.1f}%)")
print(f"Total: {len(df_train) + len(df_val) + len(df_test)}")

Train set size: 876 (59.9%)
Validation set size: 293 (20.0%)
Test set size: 293 (20.0%)
Total: 1462


In [239]:
y_train_full = df_train_full.converted.values
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

### Question 3

* Calculate the mutual information score between `y` and other categorical variables in the dataset. Use the training set only.
* Round the scores to 2 decimals using `round(score, 2)`.

Which of these variables has the biggest mutual information score?

- `lead_source`

In [240]:
mi_scores = {}
for col in categorical_columns:
    score = mutual_info_score(df_train_full[col], y_train_full)
    mi_scores[col] = round(score, 2)
    print(f"{col}: {mi_scores[col]}")

max_mi_var = max(mi_scores, key=mi_scores.get)
print(f"\nHighest mutual information: {max_mi_var} ({mi_scores[max_mi_var]})")

lead_source: 0.03
industry: 0.01
employment_status: 0.01
location: 0.0

Highest mutual information: lead_source (0.03)


### Question 4

* Now let's train a logistic regression.
* Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
* Fit the model on the training dataset.
    - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
    - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
* Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.74

In [241]:
short_num = [col for col in numeric_columns if col != 'converted']
train_dicts = df_train[categorical_columns + short_num].to_dict(orient='records')
val_dicts = df_val[categorical_columns + short_num].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1] >= 0.5
accuracy = (y_pred == y_val).mean()

print(f"Validation accuracy: {round(accuracy, 2)}")

Validation accuracy: 0.7


### Question 5 

* Let's find the least useful feature using the *feature elimination* technique.
* Train a model using the same features and parameters as in Q4 (without rounding).
* Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
* For each feature, calculate the difference between the original accuracy and the accuracy without the feature. 

Which of following feature has the smallest difference?

- `'industry'`

> **Note**: The difference doesn't have to be positive.

In [242]:
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1] >= 0.5
baseline_accuracy = (y_pred == y_val).mean()

print(f"Baseline accuracy (all features): {baseline_accuracy}")
print()

features_to_test = ['industry', 'employment_status', 'lead_score']

differences = {}
for feature in features_to_test:
    df_train_no_feature = df_train.drop(feature, axis=1)
    df_val_no_feature = df_val.drop(feature, axis=1)
    
    train_dicts_no_feat = df_train_no_feature.to_dict(orient='records')
    val_dicts_no_feat = df_val_no_feature.to_dict(orient='records')
    
    dv_no_feat = DictVectorizer(sparse=False)
    X_train_no_feat = dv_no_feat.fit_transform(train_dicts_no_feat)
    X_val_no_feat = dv_no_feat.transform(val_dicts_no_feat)
    
    model_no_feat = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_no_feat.fit(X_train_no_feat, y_train)

    y_pred_no_feat = model_no_feat.predict_proba(X_val_no_feat)[:, 1] >= 0.5
    accuracy_no_feat = (y_pred_no_feat == y_val).mean()
    
    diff = baseline_accuracy - accuracy_no_feat
    differences[feature] = diff
    
    print(f"Without '{feature}': accuracy = {accuracy_no_feat:.4f}, difference = {diff:.4f}")

min_diff_feature = min(differences, key=lambda k: abs(differences[k]))
print(f"\nFeature with smallest difference: '{min_diff_feature}' (difference = {differences[min_diff_feature]:.4f})")

Baseline accuracy (all features): 0.6996587030716723

Without 'industry': accuracy = 0.6997, difference = 0.0000
Without 'employment_status': accuracy = 0.6962, difference = 0.0034
Without 'lead_score': accuracy = 0.7065, difference = -0.0068

Feature with smallest difference: 'industry' (difference = 0.0000)


### Question 6

* Now let's train a regularized logistic regression.
* Let's try the following values of the parameter `C`: `[0.01, 0.1, 1, 10, 100]`.
* Train models using all the features as in Q4.
* Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

- 0.01

> **Note**: If there are multiple options, select the smallest `C`.

In [243]:
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

C_values = [0.01, 0.1, 1, 10, 100]
accuracies = {}

for C in C_values:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict_proba(X_val)[:, 1] >= 0.5
    accuracy = (y_pred == y_val).mean()
    accuracies[C] = round(accuracy, 3)
    
    print(f"C = {C:>6}, Accuracy = {accuracies[C]}")

best_accuracy = max(accuracies.values())
best_C = min([c for c, acc in accuracies.items() if acc == best_accuracy])

print(f"\nBest C: {best_C} with accuracy: {accuracies[best_C]}")

C =   0.01, Accuracy = 0.7
C =    0.1, Accuracy = 0.7
C =      1, Accuracy = 0.7
C =     10, Accuracy = 0.7
C =    100, Accuracy = 0.7

Best C: 0.01 with accuracy: 0.7
